### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7, 9, dan 10) ingin memiliki alat bantu **prediksi omzet harian** tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi `total_penjualan` sebuah pesanan berdasarkan atribut-atributnya.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [2]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


**SparkSession**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas11") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 09:52:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/18 09:52:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


**A. Feature Engineering** *(bobot 20%)*

Gunakan `StringIndexer` untuk mengubah kolom `nama_menu`, `cabang`, dan `waktu_pesan` menjadi indeks numerik. Gabungkan seluruh fitur (`qty`, `harga_satuan`, dan ketiga kolom hasil indexing) menjadi satu kolom `features` menggunakan `VectorAssembler`.


In [18]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler

df = spark.createDataFrame(data_tugas11)
df.show(5)

indexer_nama_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu_pesan = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

idx_menu_model = indexer_nama_menu.fit(df)
idx_cabang_model = indexer_cabang.fit(df)
idx_waktu_model = indexer_waktu_pesan.fit(df)

df_indexed = idx_menu_model.transform(df)
df_indexed = idx_cabang_model.transform(df_indexed)
df_indexed = idx_waktu_model.transform(df_indexed)

assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)
df_final.select("features", "total_penjualan").show(5, truncate=False)

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,200

**B. Train/Test Split & Pelatihan Model** *(bobot 20%)*

Bagi data menjadi 75% training dan 25% testing (`seed=101`). Latih model `LinearRegression` dengan `labelCol="total_penjualan"`.

In [19]:
from pyspark.ml.regression import LinearRegression

train, test = df_final.randomSplit([0.75, 0.25], seed=101)
lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)

print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing   : {test.count()}")
print("Model berhasil dilatih!")

Jumlah data training : 1813
Jumlah data testing   : 687
Model berhasil dilatih!


**C. Evaluasi Model** *(bobot 20%)*

Hitung dan tampilkan **RMSE** dan **R²** dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?

In [20]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)

evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")
df.agg({"total_penjualan": "avg"}).show()

RMSE : Rp 10,388
R²   : 0.9163  (91.6% variasi data berhasil dijelaskan model)
+--------------------+
|avg(total_penjualan)|
+--------------------+
|  65173.587602408494|
+--------------------+



Model ini memiliki R² sekitar 91,6% variasi data berhasil dijelaskan model, dan RMSE sebesar Rp 10.388, dibandingkan rata-rata penjualan sebesar Rp 65.174. Kesalahan prediksi rata-rata sekitar 16% dari rata-rata penjualan, sehingga model ini cukup baik untuk memprediksi omzet meskipun masih menyisakan kesalahan yang perlu diperhatikan untuk keputusan yang membutuhkan ketepatan tinggi.

**D. Eksperimen Perbandingan Fitur** *(bobot 25%)*

Latih **model kedua** yang HANYA menggunakan fitur `qty` dan `harga_satuan` (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

In [21]:
assembler_2 = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_2"
)

df_2 = assembler_2.transform(df_final)

train_2, test_2 = df_2.randomSplit([0.75, 0.25], seed=101)

lr_2 = LinearRegression(featuresCol="features_2", labelCol="total_penjualan")
model_2 = lr_2.fit(train_2)

predictions_2 = model_2.transform(test_2)
r2_2 = evaluator_r2.evaluate(predictions_2)

print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing   : {test.count()}")
print()
print(f"R²   : {r2:.4f}")
print(f"R² Model 2  : {r2_2:.4f}")
print(f"Selisih  : {r2 - r2_2:.4f}")

Jumlah data training : 1813
Jumlah data testing   : 687

R²   : 0.9163
R² Model 2  : 0.9169
Selisih  : -0.0006


R² model utama (5 fitur) adalah 0,9163, sedangkan R² model kedua yang hanya memakai qty dan harga_satuan adalah 0,9169, dengan selisih −0,0006. Jadi menambahkan informasi menu, cabang, dan waktu pesan tidak meningkatkan akurasi prediksi secara berarti, bahkan model kedua sedikit lebih tinggi. Hal ini terjadi karena total_penjualan pada dasarnya ditentukan oleh qty × harga_satuan, sedangkan menu, cabang, dan waktu tidak memengaruhinya, sehingga fitur tambahan hanya menambah noise tanpa informasi baru.

**E. Prediksi Skenario Baru** *(bobot 15%)*

Buat sebuah DataFrame baru berisi **3 skenario pesanan (data dummy)** (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan `StringIndexer` dan `VectorAssembler` yang sama seperti data training — perhatikan: gunakan model `StringIndexer` yang **sama** yang sudah di-`fit()` sebelumnya, jangan membuat `StringIndexer` baru). Tampilkan hasil prediksi `total_penjualan` untuk ketiga skenario tersebut.

In [22]:
data_baru = pd.DataFrame({
    "nama_menu": ["Kentang Goreng", "Roti Bakar", "Kentang Goreng"],
    "cabang": ["Semarang", "Magelang", "Yogyakarta"],
    "waktu_pesan": ["Malam", "Pagi", "Siang"],
    "qty": [3, 2, 5],
    "harga_satuan": [30000, 30000, 20000],
})

df_baru = spark.createDataFrame(data_baru)

df_baru = idx_menu_model.transform(df_baru)
df_baru = idx_cabang_model.transform(df_baru)
df_baru = idx_waktu_model.transform(df_baru)

df_baru = assembler.transform(df_baru)

hasil_baru = model.transform(df_baru)
hasil_baru.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(truncate=False)

+--------------+----------+-----------+---+------------+------------------+
|nama_menu     |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|Semarang  |Malam      |3  |30000       |90475.59006985578 |
|Roti Bakar    |Magelang  |Pagi       |2  |30000       |67013.12179963932 |
|Kentang Goreng|Yogyakarta|Siang      |5  |20000       |104843.39944618406|
+--------------+----------+-----------+---+------------+------------------+



**Eksplorasi 1: Melihat “kamus” StringIndexer**

In [23]:
for nama, m in [("nama_menu", idx_menu_model),
                ("cabang", idx_cabang_model),
                ("waktu_pesan", idx_waktu_model)]:
    kamus = {label: i for i, label in enumerate(m.labelsArray[0])}
    print(f"{nama:12s} -> {kamus}")

print()
df_baru.select("nama_menu", "nama_menu_idx",
               "cabang", "cabang_idx",
               "waktu_pesan", "waktu_pesan_idx").show(truncate=False)

nama_menu    -> {'Americano': 0, 'Kopi Susu': 1, 'Cappuccino': 2, 'Es Teh': 3, 'Roti Bakar': 4, 'Kentang Goreng': 5}
cabang       -> {'Magelang': 0, 'Yogyakarta': 1, 'Semarang': 2}
waktu_pesan  -> {'Pagi': 0, 'Malam': 1, 'Siang': 2, 'Sore': 3}

+--------------+-------------+----------+----------+-----------+---------------+
|nama_menu     |nama_menu_idx|cabang    |cabang_idx|waktu_pesan|waktu_pesan_idx|
+--------------+-------------+----------+----------+-----------+---------------+
|Kentang Goreng|5.0          |Semarang  |2.0       |Malam      |1.0            |
|Roti Bakar    |4.0          |Magelang  |0.0       |Pagi       |0.0            |
|Kentang Goreng|5.0          |Yogyakarta|1.0       |Siang      |2.0            |
+--------------+-------------+----------+----------+-----------+---------------+



**Eksplorasi 2: Membandingkan prediksi dengan hitungan manual**

In [24]:
from pyspark.sql.functions import col

hasil_baru \
    .withColumn("estimasi_manual", col("qty") * col("harga_satuan")) \
    .withColumn("selisih", col("prediction") - col("estimasi_manual")) \
    .select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan",
            "estimasi_manual", "prediction", "selisih") \
    .show(truncate=False)

+--------------+----------+-----------+---+------------+---------------+------------------+-----------------+
|nama_menu     |cabang    |waktu_pesan|qty|harga_satuan|estimasi_manual|prediction        |selisih          |
+--------------+----------+-----------+---+------------+---------------+------------------+-----------------+
|Kentang Goreng|Semarang  |Malam      |3  |30000       |90000          |90475.59006985578 |475.5900698557816|
|Roti Bakar    |Magelang  |Pagi       |2  |30000       |60000          |67013.12179963932 |7013.121799639324|
|Kentang Goreng|Yogyakarta|Siang      |5  |20000       |100000         |104843.39944618406|4843.399446184063|
+--------------+----------+-----------+---+------------+---------------+------------------+-----------------+



In [25]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
